# Project 10 — Local Code Explainer

## Explain Code and Detect Issues

**Goal:** Analyze code snippets to produce explanations, identify potential
issues, and suggest improvements.

**Stack:** Ollama · LangChain · Jupyter

### What You'll Learn
1. Code-focused prompting strategies
2. Line-by-line explanation generation
3. Bug detection and improvement suggestions
4. Structured vs freeform code analysis

### Prerequisites
- **Ollama** with a chat model (e.g., `qwen3:8b`)

In [ ]:
!pip install -q langchain langchain-ollama

## Step 1 — Verify Ollama Is Running

In [ ]:
import requests
OLLAMA_BASE = "http://localhost:11434"
try:
    r = requests.get(f"{OLLAMA_BASE}/api/tags", timeout=5)
    r.raise_for_status()
    models = r.json().get("models", [])
    print(f"Ollama running — {len(models)} model(s) available")
    for m in models[:5]: print(f"  - {m['name']}")
except Exception as e:
    print(f"Cannot reach Ollama at {OLLAMA_BASE}: {e}")
    print("Start Ollama first: ollama serve")

## Step 2 — Sample Code Snippets

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOllama(model="qwen3:8b", temperature=0.2)

snippets = {
    "python_sort": '''def bubble_sort(arr):
    n = len(arr)
    for i in range(n):
        for j in range(0, n-i-1):
            if arr[j] > arr[j+1]:
                arr[j], arr[j+1] = arr[j+1], arr[j]
    return arr''',
    "python_bug": '''def average(numbers):
    total = 0
    for n in numbers:
        total += n
    return total / len(numbers)  # Bug: empty list -> ZeroDivisionError''',
    "sql_query": '''SELECT u.name, COUNT(o.id) as order_count
FROM users u
LEFT JOIN orders o ON u.id = o.user_id
WHERE o.created_at > '2024-01-01'
GROUP BY u.name
HAVING order_count > 5
ORDER BY order_count DESC;''',
}
print(f"{len(snippets)} code snippets loaded")
for name in snippets: print(f"  - {name}")

## Step 3 — Line-by-Line Explanation

In [ ]:
explain_prompt = ChatPromptTemplate.from_messages([
    ("system", "Explain this code line by line. For each important line, "
     "explain what it does and why. Use beginner-friendly language."),
    ("human", "```\n{code}\n```")
])
explain_chain = explain_prompt | llm | StrOutputParser()

result = explain_chain.invoke({"code": snippets["python_sort"]})
print("BUBBLE SORT EXPLANATION:\n")
print(result)

## Step 4 — Bug Detection

In [ ]:
bug_prompt = ChatPromptTemplate.from_messages([
    ("system", "Analyze this code for bugs, edge cases, and potential issues. "
     "For each issue found: describe the bug, show the problematic line, "
     "explain the impact, and provide a fix."),
    ("human", "```\n{code}\n```")
])
bug_chain = bug_prompt | llm | StrOutputParser()

result = bug_chain.invoke({"code": snippets["python_bug"]})
print("BUG ANALYSIS:\n")
print(result)

## Step 5 — Improvement Suggestions

In [ ]:
improve_prompt = ChatPromptTemplate.from_messages([
    ("system", "Suggest improvements for this code: performance, readability, "
     "best practices, error handling. Show the improved version."),
    ("human", "```\n{code}\n```")
])
improve_chain = improve_prompt | llm | StrOutputParser()

result = improve_chain.invoke({"code": snippets["sql_query"]})
print("SQL IMPROVEMENTS:\n")
print(result)

## Step 6 — Structured Analysis

In [ ]:
struct_prompt = ChatPromptTemplate.from_messages([
    ("system", "Analyze this code and return a structured report:\n"
     "LANGUAGE: ...\nPURPOSE: one sentence\nCOMPLEXITY: O(?)\n"
     "BUGS: list or 'none'\nIMPROVEMENTS: list\nRATING: 1-10"),
    ("human", "```\n{code}\n```")
])
struct_chain = struct_prompt | llm | StrOutputParser()

for name, code_str in snippets.items():
    result = struct_chain.invoke({"code": code_str})
    print(f"\n{'='*50}")
    print(f"Snippet: {name}")
    print(result)

## Limitations

1. **Not a real linter** — may miss subtle bugs.
2. **Language support** varies by model.
3. **No execution** — analysis is static only.

## What You Learned

| Concept | What We Did |
|---|---|
| **Line-by-line** | Beginner-friendly explanations |
| **Bug detection** | Find issues + fixes |
| **Improvements** | Performance + readability |
| **Structured analysis** | Consistent report format |